In [1]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import FewShotChatMessagePromptTemplate
from langchain_openai import ChatOpenAI
import os

from sklearn.metrics import root_mean_squared_error 

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import re
import neptune

from dotenv import load_dotenv

load_dotenv()

True

# Parameters

In [32]:
post_type = 'post_travel' # 'post_travel' or 'post_abortion'
narcism_type = 'adm' # 'riv' or 'adm'
# model chosen from https://platform.openai.com/docs/models/continuous-model-upgrades
model_used = 'gpt-4o' # 'gpt-3.5-turbo-0125', 'gpt-4-1106-preview' or 'gpt-4-turbo-2024-04-09'
iterations = 10
number_of_test_posts = 10 # somewhere between 3 and 10
model_role = "You are a psychologist. Your task is to asses intelligence of the patient (Negative, Positive, Neutral) and provide a explanation for your assessment."
train_path = "../data/split/full_train.csv"
validate_path = "../data/split/full_validate.csv"

# Code

Here we used the most basic implementation, there is also option to use Dynamic few-shot prompting, but to my knowledge is not needed is this context as we have only one type of posts.

In [59]:
# Examples:
examples = [{
    "post": "I travel a lot for work, and I get to see all sorts of cool places. I didn't think the American south had as many hidden gems as it did, I think it's a beautiful region with some crazy cool people.",
    "intelligence_score": "Neutral",
    "reason": "The post primarily reflects personal experiences and subjective observations rather than insights or discussions that explicitly convey intelligence or cognitive processes. While it contains thoughtful appreciation for exploration and discovery, it does not directly emphasize intelligence-driven concepts."
},
{
    "post": "everyone has a right to bodily autonomy - banning abortion is disgusting",
    "intelligence_score": "Negative",
    "reason": "The post expresses a strong moral and emotional stance rather than focusing on rational or intellectual reasoning. The tone conveys a negative sentiment towards the concept of banning abortion, which leads to a negative intelligence score due to the emotional, rather than analytical, nature of the language used."
},
{
    "post": "I’m amazed by how AI is reshaping our world. From enhancing productivity to solving complex challenges, it’s revolutionizing various fields. I’m excited to see where this technology will take us next and how it will continue to transform our everyday lives.#TechRevolution",
    "intelligence_score": "Positive",
    "reason": "The post has a highly optimistic and forward-looking tone, emphasizing the transformative and problem-solving capabilities of AI, which aligns with a positive association with intelligence."
},
]

In [33]:
model = ChatOpenAI(model=model_used, openai_api_key=os.getenv('OPENAI_API_KEY'))


In [25]:
# This is a prompt template used to format each example.
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{post}"),
        ("ai", "score: {intelligence_score}, reason: {reason}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# print(few_shot_prompt.format())

#### Use train and validate dataset!!!

In [26]:
df = pd.read_csv(train_path)

test = df[[post_type]].iloc[4] # Test on train dataset
input = test.iloc[0]
input

'I am feeling happy and thriving with my happiness. hope you are all happy and thriving.'

In [ ]:
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", model_role), # "assistant" for gpt-o1/o3
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

final_prompt

ChatPromptTemplate(input_variables=['input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='Your task is to asses inteligence of the patient (Negative, Positive, Neutral) and provide a explanation for your assessment.')), FewShotChatMessagePromptTemplate(examples=[{'post': "I travel a lot for work, and I get to see all sorts of cool places. I didn't think the American south had as many hidden gems as it did, I think it's a beautiful region with some crazy cool people.", 'inteligence_score': 'Neutral', 'reason': 'The post primarily reflects personal experiences and subjective observations rather than insights or discussions that explicitly convey intelligence or cognitive processes. While it contains thoughtful appreciation for exploration and discovery, it does not directly emphasize intelligence-driven concepts.'}, {'post': 'everyone has a right to bodily autonomy - banning abortion is disgusting', 'inteligence_score': 'Negative', 'reason': 

In [34]:
chain = final_prompt | model

ai_message = chain.invoke({"input": input})

ai_message


AIMessage(content='score: Neutral, reason: The post expresses a positive emotional state and well-wishing towards others, but it does not engage in complex reasoning, problem-solving, or analysis that would indicate a high level of cognitive function. It is more focused on sharing personal feelings and spreading positivity.', response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 233, 'total_tokens': 290, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_4691090a87', 'finish_reason': 'stop', 'logprobs': None}, id='run-fdfa3c9b-fa5b-4a05-b111-a0b62f86c912-0')

# Analyze the results

In [49]:
# Get the score and reason from the AI response
text = ai_message.content
score = re.search('score: (.+?),', text).group(1)
reason = re.search("reason: .*", text).group(0)
# Cut reason from reason:
reason = reason[8:]
print(f"Score: {score}")
print(f"Reason: {reason}")


Score: Neutral
Reason: The post expresses a positive emotional state and well-wishing towards others, but it does not engage in complex reasoning, problem-solving, or analysis that would indicate a high level of cognitive function. It is more focused on sharing personal feelings and spreading positivity.


# Implementation

In [ ]:
# functions

# get random x posts
def get_random_x_posts(path, post_type, number_of_posts):

    df = pd.read_csv(path)
    example = df[[post_type]].sample(number_of_posts)
    example = example.to_dict(orient='records')

    # Change the value name
    for i in range(len(example)):
        example[i]['post'] = example[i].pop(post_type)

    return example

# create a few shot prompt
def create_few_shot_prompt(examples):
    example_prompt = ChatPromptTemplate.from_messages(
        [
            ("human", "{post}"),
            ("ai", "score: {intelligence_score}, reason: {reason}"),
        ]
    )
    few_shot_prompt = FewShotChatMessagePromptTemplate(
        example_prompt=example_prompt,
        examples=examples,
    )
    return few_shot_prompt

# create a final prompt
def create_final_prompt(few_shot_prompt,model_role):
    final_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", model_role),
            few_shot_prompt,
            ("human", "{input}"),
        ]
    )
    return final_prompt

# Get the score and reason from the AI response
def get_score_and_reason(text):
    match_score = re.search('score: (.+?),', text)
    match_reason = re.search("reason: .*", text)
    if match_score and match_reason:
        score = re.search('score: (.+?),', text).group(1)
        reason = re.search("reason: .*", text).group(0)
        # Cut reason from reason:
        reason = reason[8:]
    else:
        print(f"Wrong input: {text}")
        return None, None

    return score, reason

# get the response
def get_response(final_prompt, model, input):
    chain = final_prompt | model
    ai_message = chain.invoke({"input": input})
    response = ai_message.content
    return response

# Prepare the dataframe

In [81]:
post_type = 'post_abortion' # 'post_travel' or 'post_abortion'
number_of_posts = 25

In [82]:
test_df = pd.read_csv(train_path)
testset = test_df[[post_type]].sample(number_of_posts)

#output = pd.DataFrame(columns=['post', 'score', 'reason'])
output = []
problems = []

for i in range(testset.shape[0]):
    few_shot_prompt = create_few_shot_prompt(examples)
    input = testset.iloc[i]
    final_prompt = create_final_prompt(few_shot_prompt,model_role)
    response_str = get_response(final_prompt, model, input.get(post_type))
    score, reason = get_score_and_reason(response_str)

    if score is None or reason is None:
        # Save the prompt that caused the error
        row_to_add = {'post': input.get(post_type), 'post_type': post_type, 'model_role': model_role, 'date': pd.Timestamp.now()}
        problems.append(row_to_add)
    else: 
        row_to_add = {'post': input.get(post_type), 'score': score, 'reason': reason}
        output.append(row_to_add)

In [83]:
output_df = pd.DataFrame(output)
print(problems)
output_df.head()

[]


,post,score,reason
0,Banning abortion does not stop people from get...,Positive,The statement presents a logical argument rega...
1,The Pro-Life movement has grown large enough t...,Neutral,The post reflects personal beliefs and excitem...
2,Completely insane and very frightening. I woul...,Negative,The statement conveys a strong emotional react...
3,Time to move to a different country.,Neutral,The statement is brief and provides limited co...
4,"If you don’t agree with abortion, then don’t h...",Neutral,The post reflects a personal viewpoint advocat...


In [84]:
post_type = 'post_travel' # 'post_travel' or 'post_abortion'
number_of_posts = 25

In [85]:
test_df = pd.read_csv(train_path)
testset = test_df[[post_type]].sample(number_of_posts)

#output = pd.DataFrame(columns=['post', 'score', 'reason'])
output = []
problems = []

for i in range(testset.shape[0]):
    few_shot_prompt = create_few_shot_prompt(examples)
    input = testset.iloc[i]
    final_prompt = create_final_prompt(few_shot_prompt,model_role)
    response_str = get_response(final_prompt, model, input.get(post_type))
    score, reason = get_score_and_reason(response_str)

    if score is None or reason is None:
        # Save the prompt that caused the error
        row_to_add = {'post': input.get(post_type), 'post_type': post_type, 'model_role': model_role, 'date': pd.Timestamp.now()}
        problems.append(row_to_add)
    else: 
        row_to_add = {'post': input.get(post_type), 'score': score, 'reason': reason}
        output.append(row_to_add)

In [86]:
# concatenate the output
full_output_df = pd.concat([output_df, pd.DataFrame(output)])
print(problems)
full_output_df.head()

[]


,post,score,reason
0,Banning abortion does not stop people from get...,Positive,The statement presents a logical argument rega...
1,The Pro-Life movement has grown large enough t...,Neutral,The post reflects personal beliefs and excitem...
2,Completely insane and very frightening. I woul...,Negative,The statement conveys a strong emotional react...
3,Time to move to a different country.,Neutral,The statement is brief and provides limited co...
4,"If you don’t agree with abortion, then don’t h...",Neutral,The post reflects a personal viewpoint advocat...


In [ ]:
# Save the output
full_output_df.to_csv("../data/reporting/intelligence_test.csv", index = False)